# Damped Pendulum Animation

Simulate and animate a **damped** simple pendulum:
- **Newton's law with friction**: $\ddot{\theta} = -\frac{g}{L} \sin\theta - c \,\dot{\theta}$
- **Numerical integration** via `scipy.integrate.solve_ivp`
- **Animation** via `matplotlib.animation.FuncAnimation`

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

print("Libraries loaded.")

## 1. Damped Pendulum Equation

With viscous damping $c$ (units 1/s):

$$
\ddot{\theta} = -\frac{g}{L} \sin\theta - c \,\dot{\theta}
$$

First-order system:

$$
\begin{cases}
\dot{\theta} = \omega \\
\dot{\omega} = -\dfrac{g}{L} \sin\theta - c \,\omega
\end{cases}
$$

In [ ]:
g = 9.81
L = 1.0
c = 0.5

def pendulum_ode(t, y):
    theta, omega = y
    dtheta = omega
    domega = -(g / L) * np.sin(theta) - c * omega
    return [dtheta, domega]

## 2. Simulate the Trajectory

Start from $\theta_0 = 170^\circ$ with zero initial velocity.

In [ ]:
theta0 = np.deg2rad(170)
omega0 = 0.0
t_span = (0, 10)
t_eval = np.linspace(0, 10, 300)

sol = solve_ivp(
    pendulum_ode, t_span, [theta0, omega0],
    t_eval=t_eval, method='RK45', rtol=1e-8
)

theta_hist = sol.y[0]
t_hist = sol.t

print(f"Simulated {len(t_hist)} time steps.")

## 3. Compare Different Damping Levels

Simulate the same pendulum with several damping coefficients $c$ and plot $\theta(t)$.

In [ ]:
c_values = [0.0, 0.2, 0.5, 1.0, 2.0]
colors = ['C0', 'C1', 'C2', 'C3', 'C4']

fig0, ax0 = plt.subplots(figsize=(6, 3))

for c_val, color in zip(c_values, colors):
    def ode(t, y):
        theta, omega = y
        return [omega, -(g / L) * np.sin(theta) - c_val * omega]
    sol_c = solve_ivp(ode, t_span, [theta0, omega0],
                      t_eval=t_eval, method='RK45', rtol=1e-8)
    ax0.plot(t_eval, np.rad2deg(sol_c.y[0]), color=color,
             label=f"$c = {c_val}$")

ax0.set_xlabel("Time (s)")
ax0.set_ylabel(r"$\theta$ (deg)")
ax0.set_title("Effect of Damping on Pendulum Motion")
ax0.legend()
ax0.grid(True, alpha=0.3)
plt.show()

## 4. Animate the Damped Pendulum

The bob's position is $(x, y) = (L \sin\theta, -L \cos\theta)$.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(-1.2 * L, 1.2 * L)
ax.set_ylim(-1.2 * L, 1.2 * L)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title(f"Damped Pendulum  ($c = {c}$)")

line, = ax.plot([], [], 'o-', lw=2, markersize=8)
trace, = ax.plot([], [], '.', alpha=0.3, markersize=1)

x_hist = L * np.sin(theta_hist)
y_hist = -L * np.cos(theta_hist)

def init():
    line.set_data([], [])
    trace.set_data([], [])
    return line, trace

def animate(i):
    x = L * np.sin(theta_hist[i])
    y = -L * np.cos(theta_hist[i])
    line.set_data([0, x], [0, y])
    trace.set_data(x_hist[:i], y_hist[:i])
    return line, trace

ani = FuncAnimation(fig, animate, frames=len(t_hist),
                    init_func=init, blit=True, interval=30)

plt.close(fig)
HTML(ani.to_jshtml())

## 5. Phase Portrait

Plot $\theta$ vs $\dot{\theta}$ for the damped case.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(5, 4))
ax2.plot(np.rad2deg(theta_hist), np.rad2deg(sol.y[1]), color='C2')
ax2.set_xlabel(r"$\theta$ (deg)")
ax2.set_ylabel(r"$\dot{\theta}$ (deg/s)")
ax2.set_title("Phase Portrait (damped)")
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='gray', lw=0.5)
ax2.axvline(0, color='gray', lw=0.5)
plt.show()